# Comparação de stacking e votação com os modelos do projeto

Este notebook compara quatro formas de combinar **os mesmos três artefatos pré-treinados**: CatBoost, LightGBM e XGBoost. Usa os dados OBT locais, `is_fraud` como target e a regra financeira já implementada no projeto.

## O que cada método faz

**Stacking** aprende uma segunda camada: os classificadores-base produzem saídas e um meta-classificador (aqui, regressão logística) aprende como combiná-las. **Votação soft** não aprende uma segunda camada; calcula a média das probabilidades dos modelos-base e escolhe a classe pela maior probabilidade.

| Método | Combinação | Diferença prática nesta avaliação |
|---|---|---|
| `sklearn.ensemble.StackingClassifier` | Meta-modelo aprende a combinar probabilidades. | `cv="prefit"` reutiliza os modelos já ajustados e ajusta somente a regressão logística nas previsões do período de meta-ajuste. |
| `mlxtend.classifier.StackingClassifier` | Também aprende um meta-modelo. | `use_probas=True`, `drop_proba_col="first"` mantém a probabilidade da classe 1 como meta-feature; `fit_base_estimators=False` deixa os três artefatos congelados. |
| `sklearn.ensemble.VotingClassifier` | Média das probabilidades (soft voting), com pesos iguais. | Seu `fit` padrão clona e ajusta estimadores; o projeto já reutiliza modelos pré-ajustados injetando `estimators_`, padrão mantido aqui. |
| `mlxtend.classifier.EnsembleVoteClassifier` | Média das probabilidades (soft voting), com pesos iguais. | Permite declarar estimadores pré-ajustados com `fit_base_estimators=False` e `use_clones=False`. |



In [1]:
#!pip install -U xgboost -> caso o importa fique dando eroos 

In [2]:
from pathlib import Path
import importlib.metadata as metadata
import sys
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from mlxtend.classifier import EnsembleVoteClassifier, StackingClassifier as MlxtendStackingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

EVALUATION_DIR = Path.cwd()
if not (EVALUATION_DIR / "evaluate.py").exists():
    EVALUATION_DIR = EVALUATION_DIR / "program" / "evaluation"
if not (EVALUATION_DIR / "evaluate.py").exists():
    raise FileNotFoundError("Não foi possível localizar program/evaluation/evaluate.py")
sys.path.insert(0, str(EVALUATION_DIR))
from evaluate import Evaluate



In [3]:
import os
print(os.getcwd())

pathModel = Path(os.getcwd()).parent / "models"
print(pathModel)

c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\program\evaluation
c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\program\models


In [4]:
MODEL_DIR = pathModel

catboost_model = CatBoostClassifier()
catboost_model.load_model(str(MODEL_DIR / "catboost.cbm"), format="cbm")

lightgbm_model = lgb.Booster(model_file=str(MODEL_DIR / "lightgbm.txt"))

xgboost_model = xgb.XGBClassifier(enable_categorical=True, n_jobs=4)
xgboost_model.load_model(str(MODEL_DIR / "xgboost.json"))
xgboost_booster = xgboost_model.get_booster()

FEATURES =  [
    "tx_amount", "hour", "day_week", "terminal_age_days",
    "terminal_fraud_rate", "card_fraud_rate", "card_distinct_terminals",
    "terminal_new_for_card", "card_tx_count_1h", "card_tx_count_24h",
    "card_distinct_terminals_24h", "seconds_since_last_card_tx",
    "terminal_tx_count_1h", "terminal_tx_count_24h",
    "terminal_distinct_cards_24h", "seconds_since_last_terminal_tx",
    "card_history_count_30d", "card_amount_mean_30d",
    "card_amount_std_30d", "amount_z_score_30d", "terminal_category",
]

TARGET = "is_fraud"

CATBOOST_PATH = MODEL_DIR / "catboost.cbm"
LIGHTGBM_PATH = MODEL_DIR / "lightgbm.txt"
XGBOOST_PATH = MODEL_DIR / "xgboost.json"
model_paths = {
    "CatBoost": CATBOOST_PATH,
    "LightGBM": LIGHTGBM_PATH,
    "XGBoost": XGBOOST_PATH,
}
print("Os três modelos carregaram com 21 features alinhadas e classe 1 disponível.")


Os três modelos carregaram com 21 features alinhadas e classe 1 disponível.


In [5]:
TRAIN_PATH = Path(os.getcwd()).parent.parent  /  "data" / "processed" / "obt_train.parquet"
VALIDATION_PATH = Path(os.getcwd()).parent.parent / "data" / "processed" / "obt_validation.parquet"
DATETIME = "tx_datetime"
AMOUNT = "tx_amount"
print(f"Arquivos de treino e validação: {TRAIN_PATH}, {VALIDATION_PATH}")

obtTrain = pd.read_parquet(TRAIN_PATH)
obtValidation = pd.read_parquet(VALIDATION_PATH)


Arquivos de treino e validação: c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\data\processed\obt_train.parquet, c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\data\processed\obt_validation.parquet


In [6]:
base_models = [
    ("catboost", catboost_model),
    ("lightgbm", lightgbm_model),
    ("xgboost", xgboost_model),
]
print("Base models loaded: ", [name for name, _ in base_models])

Base models loaded:  ['catboost', 'lightgbm', 'xgboost']


In [7]:
from sklearn.base import BaseEstimator, ClassifierMixin

# Classe para lidar com os modelos baixados(gerava um objeto de baixo nivel) que não possuem a interface do scikit-learn par astacking
class ClassificadorPrefit(ClassifierMixin, BaseEstimator):
    def __init__(self, model, tipo, lgb_categories=None, xgb_categories=None):
        self.model = model
        self.tipo = tipo
        self.lgb_categories = lgb_categories
        self.xgb_categories = xgb_categories

        # Indica ao sklearn que este adaptador envolve um modelo já ajustado.
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        raise RuntimeError("Este adaptador é apenas para modelos pré-ajustados.")

    def predict_proba(self, X):
        X_model = X.copy()

        if self.tipo == "lightgbm":
            valores = (
                X_model["terminal_category"]
                .astype("object")
                .fillna("SEM_CATEGORIA")
            )
            X_model["terminal_category"] = pd.Categorical(
                valores,
                categories=self.lgb_categories,
            )
            prob_fraude = self.model.predict(X_model)

        elif self.tipo == "xgboost":
            valores = (
                X_model["terminal_category"]
                .astype("object")
                .fillna("SEM_CATEGORIA")
            )
            X_model["terminal_category"] = pd.Categorical(
                valores,
                categories=self.xgb_categories,
            )
            prob_fraude = self.model.predict_proba(X_model)[:, 1]

        else:  # CatBoost
            prob_fraude = self.model.predict_proba(X_model)[:, 1]

        return np.column_stack([1 - prob_fraude, prob_fraude])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 1. StackingClassifier do scikit-learn

`cv="prefit"` é apropriado porque os modelos-base vêm carregados dos arquivos. A validação é dividida cronologicamente: 50% para ajustar o meta-modelo, 25% para escolher o corte que maximiza o lucro e 25% para medir o resultado final sem reajustar o corte.

In [8]:
lgb_categories = lightgbm_model.feature_name()

xgb_categories = pd.Categorical(
    obtTrain["terminal_category"]
    .astype("object")
    .fillna("SEM_CATEGORIA")
).categories

estimators = [
    ("catboost", ClassificadorPrefit(catboost_model, "catboost")),
    (
        "lightgbm",
        ClassificadorPrefit(
            lightgbm_model,
            "lightgbm",
            lgb_categories=lgb_categories,
        ),
    ),
    (
        "xgboost",
        ClassificadorPrefit(
            xgboost_model,
            "xgboost",
            xgb_categories=xgb_categories,
        ),
    ),
]
sklearn_stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    stack_method="predict_proba",
    cv="prefit",
    passthrough=False,
)

# Divide a validação em ordem cronológica: meta-ajuste, calibração do corte e teste final.
obtValidation = obtValidation.sort_values("tx_datetime").reset_index(drop=True)
limite_meta = int(len(obtValidation) * 0.50)
limite_calibracao = int(len(obtValidation) * 0.75)
obtMeta = obtValidation.iloc[:limite_meta].copy()
obtCalibracao = obtValidation.iloc[limite_meta:limite_calibracao].copy()
obtPredict = obtValidation.iloc[limite_calibracao:].copy()

X_meta = obtMeta[FEATURES]
y_meta = obtMeta[TARGET]
X_calibracao = obtCalibracao[FEATURES]
y_calibracao = obtCalibracao[TARGET]
X_predict = obtPredict[FEATURES]
y_predict = obtPredict[TARGET]

sklearn_stacking.fit(X_meta, y_meta)
print("StackingClassifier do scikit-learn ajustado; calibração e teste permanecem separados.")


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


StackingClassifier do scikit-learn ajustado; calibração e teste permanecem separados.


In [9]:
def probabilidades_fraude(modelo, x, isvoting=False):
    if isvoting:
        modelo.estimators_ = [estimador for _, estimador in modelo.estimators]
        modelo.le_ = LabelEncoder().fit([0, 1])
        modelo.classes_ = np.array([0, 1])
    return modelo.predict_proba(x)[:, 1]


def encontrar_melhor_corte(modelo, x_calibracao, y_calibracao, isvoting=False):
    probabilidades = probabilidades_fraude(modelo, x_calibracao, isvoting)
    avaliador = Evaluate(
        y_true=y_calibracao.to_numpy(),
        probabilidades=probabilidades,
        valores=x_calibracao[AMOUNT].to_numpy(),
        comissao=0.03,
    )
    return avaliador.buscar_corte()


In [10]:
def maior_lucro_possivel(x, y):
    # Com 100% de acerto, todas as legítimas são aprovadas e todas as fraudes recusadas.
    return float(0.03 * x.loc[y.eq(0), AMOUNT].sum())


def exibir_lucro(lucro, x, y):
    maximo = maior_lucro_possivel(x, y)
    print(
        "Lucro encontrado / maior lucro possível (100% de acerto): "
        f"R$ {lucro:,.2f} / R$ {maximo:,.2f}"
    )


def gerarResiltado(modelo, x, y, corte, isvoting=False):
    probabilidades = probabilidades_fraude(modelo, x, isvoting)
    avaliador = Evaluate(
        y_true=y.to_numpy(),
        probabilidades=probabilidades,
        valores=x[AMOUNT].to_numpy(),
        comissao=0.03,
    )
    # O enunciado recusa transações cuja probabilidade de fraude excede o corte.
    y_pred = (probabilidades > corte).astype(int)
    dfValidacaoPredita = pd.DataFrame({
        "tx_amount": x[AMOUNT],
        "is_fraud": y,
        "preditada": y_pred,
        "probabilidade": probabilidades,
    }, index=x.index)
    lucro = avaliador.calcular_lucro(corte)
    return dfValidacaoPredita, lucro,corte


In [11]:
corte_sklearn_stacking, lucro_calibracao_sklearn_stacking = encontrar_melhor_corte(
    sklearn_stacking, X_calibracao, y_calibracao
)
dfValidacaoPredita, lucro,corte = gerarResiltado(
    sklearn_stacking, X_predict, y_predict, corte_sklearn_stacking
)
confusion_matrix = pd.crosstab(
    y_predict, dfValidacaoPredita["preditada"],
    rownames=["Actual"], colnames=["Predicted"]
)
print("Corte escolhido na calibração:", corte_sklearn_stacking)
exibir_lucro(lucro, X_predict, y_predict)
print("Matriz de Confusão:")
print(confusion_matrix)
dfValidacaoPredita


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


Corte escolhido na calibração: 0.02774640787627098
Lucro encontrado / maior lucro possível (100% de acerto): R$ 485,430.28 / R$ 591,040.62
Matriz de Confusão:
Predicted       0     1
Actual                 
False      259985   757
True         1389  1632


,tx_amount,is_fraud,preditada,probabilidade
791289,33.869999,False,0,0.007528
791290,42.299999,False,0,0.007632
791291,77.089996,False,0,0.007797
791292,11.700000,False,0,0.007407
791293,3.060000,False,0,0.007411
...,...,...,...,...
1055047,61.299999,False,0,0.007395
1055048,5.510000,False,0,0.007546
1055049,59.549999,False,0,0.007485
1055050,53.150002,False,0,0.007545


In [12]:
falsoPositivo = dfValidacaoPredita.loc[
    (dfValidacaoPredita["is_fraud"] == 0)
    & (dfValidacaoPredita["preditada"] == 1)
]
falsoPositivo


,tx_amount,is_fraud,preditada,probabilidade
791696,406.489990,False,1,0.038576
792012,6.570000,False,1,0.190929
792906,51.540001,False,1,0.063864
794213,128.750000,False,1,0.161779
795076,24.059999,False,1,0.208417
...,...,...,...,...
1051966,73.980003,False,1,0.047590
1053431,67.949997,False,1,0.175615
1053626,39.700001,False,1,0.035418
1054679,36.580002,False,1,0.053555


## 2. StackingClassifier do mlxtend

Este segundo stacking também aprende uma regressão logística sobre probabilidades-base. `average_probas=False` mantém uma coluna por classificador e `drop_proba_col="first"` elimina a coluna da classe 0, redundante numa classificação binária. `fit_base_estimators=False` impede que os modelos carregados sejam treinados novamente.


In [13]:
classifiers_mlxtend = [modelo for _, modelo in estimators]
mlxtend_stacking = MlxtendStackingClassifier(
    classifiers=classifiers_mlxtend,
    meta_classifier=LogisticRegression(max_iter=1000),
    use_probas=True,
    average_probas=False,
    drop_proba_col="first",
    fit_base_estimators=False,
    use_clones=False,
)
mlxtend_stacking.fit(X_meta, y_meta)
print("StackingClassifier do mlxtend ajustado; estimadores-base mantidos pré-ajustados.")


c:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\program\.venv\Lib\site-packages\mlxtend\classifier\stacking_classification.py:173: UserWarning: fit_base_estimators=False enforces use_clones to be `False`
  warnings.warn(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


StackingClassifier do mlxtend ajustado; estimadores-base mantidos pré-ajustados.


In [14]:
corte_mlxtend_stacking, lucro_calibracao_mlxtend_stacking = encontrar_melhor_corte(
    mlxtend_stacking, X_calibracao, y_calibracao
)
dfValidacao, lucro,corte = gerarResiltado(
    mlxtend_stacking, X_predict, y_predict, corte_mlxtend_stacking
)
confusion_matrix = pd.crosstab(
    y_predict, dfValidacao["preditada"],
    rownames=["Actual"], colnames=["Predicted"]
)
print("Corte escolhido na calibração:", corte_mlxtend_stacking)
exibir_lucro(lucro, X_predict, y_predict)
print("Matriz de Confusão:")
print(confusion_matrix)


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


Corte escolhido na calibração: 0.02774640787627098
Lucro encontrado / maior lucro possível (100% de acerto): R$ 485,430.28 / R$ 591,040.62
Matriz de Confusão:
Predicted       0     1
Actual                 
False      259985   757
True         1389  1632


In [15]:
def calcularPeso(model1, model2, model3):
    # Define pesos com base no lucro máximo obtido somente no período de calibração.
    resultados = [
        encontrar_melhor_corte(modelo, X_calibracao, y_calibracao)[1]
        for modelo in (model1, model2, model3)
    ]
    total = sum(resultados)
    if total <= 0:
        return [1, 1, 1]
    return [lucro / total for lucro in resultados]


## 3. VotingClassifier do scikit-learn

Soft voting calcula a média simples das três distribuições de probabilidade.


In [33]:
model1 = estimators[0][1]
model2 = estimators[1][1]
model3 = estimators[2][1]
pesos = calcularPeso(model1,model2,model3)
print(pesos)
sklearn_voting = VotingClassifier(
        estimators=list(estimators),
        voting="soft",
        weights=pesos,
    )
sklearn_voting.estimators_ = estimators
sklearn_voting.le_ = LabelEncoder().fit(np.array([0, 1]))
sklearn_voting.classes_ = np.array([0, 1])
#print("VotingClassifier do scikit-learn configurado como soft voting pré-ajustado, pesos iguais.")


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


[0.33552268433094085, 0.32909175371084776, 0.33538556195821134]


In [17]:
corte_sklearn_voting, lucro_calibracao_sklearn_voting = encontrar_melhor_corte(
    sklearn_voting, X_calibracao, y_calibracao, True
)
dfValidacao, lucro,corte = gerarResiltado(
    sklearn_voting, X_predict, y_predict, corte_sklearn_voting, True
)
confusion_matrix = pd.crosstab(
    y_predict, dfValidacao["preditada"],
    rownames=["Actual"], colnames=["Predicted"]
)
print("Corte escolhido na calibração:", corte_sklearn_voting)
exibir_lucro(lucro, X_predict, y_predict)
print("Matriz de Confusão:")
print(confusion_matrix)


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


Corte escolhido na calibração: 0.06517909266950854
Lucro encontrado / maior lucro possível (100% de acerto): R$ 485,495.50 / R$ 591,040.62
Matriz de Confusão:
Predicted       0     1
Actual                 
False      259888   854
True         1383  1638


## 4. EnsembleVoteClassifier do mlxtend

Também faz soft voting com pesos uniformes. O mlxtend oferece `fit_base_estimators=False` e `use_clones=False` diretamente para manter os artefatos-base sem ajuste adicional.


In [18]:
clfs = [modelo for _, modelo in estimators]
pesos = calcularPeso(model1,model2,model3)
mlxtend_voting = EnsembleVoteClassifier(
    clfs=clfs,
    voting="soft",
    weights=pesos,
    fit_base_estimators=False,
    use_clones=False,
)
mlxtend_voting.fit(X_meta, y_meta)
print("EnsembleVoteClassifier do mlxtend ajustado sem refazer os modelos-base.")


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


EnsembleVoteClassifier do mlxtend ajustado sem refazer os modelos-base.


In [19]:
corte_mlxtend_voting, lucro_calibracao_mlxtend_voting = encontrar_melhor_corte(
    mlxtend_voting, X_calibracao, y_calibracao
)
dfValidacao, lucro,corte = gerarResiltado(
    mlxtend_voting, X_predict, y_predict, corte_mlxtend_voting
)
confusion_matrix = pd.crosstab(
    y_predict, dfValidacao["preditada"],
    rownames=["Actual"], colnames=["Predicted"]
)
print("Corte escolhido na calibração:", corte_mlxtend_voting)
exibir_lucro(lucro, X_predict, y_predict)
print("Matriz de Confusão:")
print(confusion_matrix)


C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


Corte escolhido na calibração: 0.06517909266950854
Lucro encontrado / maior lucro possível (100% de acerto): R$ 485,495.50 / R$ 591,040.62
Matriz de Confusão:
Predicted       0     1
Actual                 
False      259888   854
True         1383  1638


## Comparação com os modelos sozinho e stacking/votação
Como os dados de testes são menores para stacking e votação, a comparação é feita com os modelos individuais ajustados no período de teste. A tabela abaixo mostra o lucro líquido médio por modelo e método de combinação.

In [20]:
# Modelos individuais
catboost = estimators[0][1]
lightgbm = estimators[1][1]
xgboost = estimators[2][1]

corte_catboost, lucro_calibracao_catboost = encontrar_melhor_corte(
    catboost, X_calibracao, y_calibracao
)
corte_lightgbm, lucro_calibracao_lightgbm = encontrar_melhor_corte(
    lightgbm, X_calibracao, y_calibracao
)
corte_xgboost, lucro_calibracao_xgboost = encontrar_melhor_corte(
    xgboost, X_calibracao, y_calibracao
)

modelosIndividuais = {
    "CatBoost": (catboost, corte_catboost),
    "LightGBM": (lightgbm, corte_lightgbm),
    "XGBoost": (xgboost, corte_xgboost),
}

resultado_catboost = gerarResiltado(
    catboost, X_predict, y_predict, corte_catboost
)
resultado_lightgbm = gerarResiltado(
    lightgbm, X_predict, y_predict, corte_lightgbm
)
resultado_xgboost = gerarResiltado(
    xgboost, X_predict, y_predict, corte_xgboost
)

resultadosIndividuais = {
    "CatBoost": resultado_catboost,
    "LightGBM": resultado_lightgbm,
    "XGBoost": resultado_xgboost,
}

melhorModeloResultado = max(
    resultadosIndividuais.items(),
    key=lambda item: item[1][1],
)
melhorModeloNome = melhorModeloResultado[0]
melhorModeloLucro = melhorModeloResultado[1][1]
melhorModelo, melhorModeloCorte = modelosIndividuais[melhorModeloNome]


# Modelos Stacking
modelosStacking = {
    "StackingClassifier (sklearn)": sklearn_stacking,
    "StackingClassifier (mlxtend)": mlxtend_stacking,
}

resultadosStacking = {
    "StackingClassifier (sklearn)": gerarResiltado(
        sklearn_stacking, X_predict, y_predict, corte_sklearn_stacking
    ),
    "StackingClassifier (mlxtend)": gerarResiltado(
        mlxtend_stacking, X_predict, y_predict, corte_mlxtend_stacking
    ),
}

melhorStackingResultado = max(
    resultadosStacking.items(),
    key=lambda item: item[1][1],
)
melhorStackingName = melhorStackingResultado[0]
melhorStackingLucro = melhorStackingResultado[1][1]
melhorStacking = modelosStacking[melhorStackingName]


# Modelos Voting
modelosVoting = {
    "VotingClassifier (sklearn)": sklearn_voting,
    "EnsembleVoteClassifier (mlxtend)": mlxtend_voting,
}

resultadosVoting = {
    "VotingClassifier (sklearn)": gerarResiltado(
        sklearn_voting, X_predict, y_predict, corte_sklearn_voting, True
    ),
    "EnsembleVoteClassifier (mlxtend)": gerarResiltado(
        mlxtend_voting, X_predict, y_predict, corte_mlxtend_voting
    ),
}

melhorVotingResultado = max(
    resultadosVoting.items(),
    key=lambda item: item[1][1],
)
melhorVotingName = melhorVotingResultado[0]
melhorVotingLucro = melhorVotingResultado[1][1]
melhorVoting = modelosVoting[melhorVotingName]


# Exibir os vencedores
print(
    "Melhor modelo individual:",
    melhorModeloNome,
    "com lucro de R$ {:,.2f}".format(melhorModeloLucro),
)
print(
    "Melhor modelo de Stacking:",
    melhorStackingName,
    "com lucro de R$ {:,.2f}".format(melhorStackingLucro),
)
print(
    "Melhor modelo de Voting:",
    melhorVotingName,
    "com lucro de R$ {:,.2f}".format(melhorVotingLucro),
)

C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dty

Melhor modelo individual: CatBoost com lucro de R$ 485,419.97
Melhor modelo de Stacking: StackingClassifier (sklearn) com lucro de R$ 485,430.28
Melhor modelo de Voting: VotingClassifier (sklearn) com lucro de R$ 485,495.50


In [21]:
def gerarMatrizConfusao(modelo, x, y, corte, isvoting=False):
    dfValidacao, lucro, corte_usado = gerarResiltado(
        modelo, x, y, corte, isvoting
    )

    return pd.crosstab(
        y,
        dfValidacao["preditada"],
        rownames=["Actual"],
        colnames=["Predicted"],
    )


confusion_matrices = {
    "melhorModeloIndividual": gerarMatrizConfusao(
        melhorModelo,
        X_predict,
        y_predict,
        melhorModeloCorte,
    ),
    "melhorStacking": gerarMatrizConfusao(
        melhorStacking,
        X_predict,
        y_predict,
        corte_sklearn_stacking
        if melhorStackingName == "StackingClassifier (sklearn)"
        else corte_mlxtend_stacking,
    ),
    "melhorVoting": gerarMatrizConfusao(
        melhorVoting,
        X_predict,
        y_predict,
        corte_sklearn_voting
        if melhorVotingName == "VotingClassifier (sklearn)"
        else corte_mlxtend_voting,
        isvoting=(melhorVotingName == "VotingClassifier (sklearn)"),
    ),
}

print("Matriz de Confusão do melhor modelo individual:")
print(confusion_matrices["melhorModeloIndividual"])

print("Matriz de Confusão do melhor modelo de Stacking:")
print(confusion_matrices["melhorStacking"])

print("Matriz de Confusão do melhor modelo de Voting:")
print(confusion_matrices["melhorVoting"])
#lembrando que 0 é uma transação legítima e 1 é uma transação fraudulenta.

C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(
C:\Users\lf446\AppData\Local\Temp\ipykernel_17936\2044132144.py:26: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_model["terminal_category"] = pd.Categorical(


Matriz de Confusão do melhor modelo individual:
Predicted       0     1
Actual                 
False      260001   741
True         1390  1631
Matriz de Confusão do melhor modelo de Stacking:
Predicted       0     1
Actual                 
False      259985   757
True         1389  1632
Matriz de Confusão do melhor modelo de Voting:
Predicted       0     1
Actual                 
False      259888   854
True         1383  1638


In [22]:
nomes = {
    "melhorModeloIndividual": "Individual",
    "melhorStacking": "Stacking",
    "melhorVoting": "Voting",
}

for chave, nome in nomes.items():
    matriz = confusion_matrices[chave]

    total_fraudes = matriz.iloc[-1].sum()
    fraudes_encontradas = matriz.iloc[-1, -1]

    print(f"{nome}: {fraudes_encontradas}/{total_fraudes} fraudes encontradas")

Individual: 1631/3021 fraudes encontradas
Stacking: 1632/3021 fraudes encontradas
Voting: 1638/3021 fraudes encontradas


In [31]:
x = obtPredict[FEATURES]
y = obtPredict[TARGET]

probabilidades = xgboost.predict_proba(x)[:, 1]

resultado = obtPredict[
    ["transaction_id", "tx_datetime", "tx_amount", "card_id", "terminal_id"]
].copy()

resultado["is_fraud"] = y
resultado["probabilidade"] = probabilidades
resultado["preditada"] = (resultado["probabilidade"] > corte_xgboost).astype(int)

acerto = resultado.loc[
    (resultado["preditada"] == 1) & (resultado["is_fraud"] == 1)
].sort_values("probabilidade", ascending=False).head(10)

print(acerto.to_json(orient="records", date_format="iso", indent=2))

[
  {
    "transaction_id":4480278,
    "tx_datetime":"2018-05-31T18:34:43.000",
    "tx_amount":53.0299987793,
    "card_id":3540768564492554,
    "terminal_id":1387,
    "is_fraud":true,
    "probabilidade":0.8969577551,
    "preditada":1
  },
  {
    "transaction_id":4480245,
    "tx_datetime":"2018-05-31T18:32:55.000",
    "tx_amount":44.9000015259,
    "card_id":3507491850582407,
    "terminal_id":1289,
    "is_fraud":true,
    "probabilidade":0.8969577551,
    "preditada":1
  },
  {
    "transaction_id":4480102,
    "tx_datetime":"2018-05-31T18:24:40.000",
    "tx_amount":39.8899993896,
    "card_id":4717862089278,
    "terminal_id":1649,
    "is_fraud":true,
    "probabilidade":0.8969577551,
    "preditada":1
  },
  {
    "transaction_id":4479961,
    "tx_datetime":"2018-05-31T18:19:16.000",
    "tx_amount":37.5600013733,
    "card_id":4168120750006997,
    "terminal_id":1289,
    "is_fraud":true,
    "probabilidade":0.8969577551,
    "preditada":1
  },
  {
    "transaction_id":